In [1]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

NUM_CLASSES = 20
IMAGE_EMBEDDING_DIM = 2048
TEXT_EMBEDDING_DIM = 768

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.14.0
Device: mps


1. Build the Image Encoder

In [2]:
weights = models.ResNet50_Weights.DEFAULT

resnet = models.resnet50(
    weights=weights
)

resnet.fc = nn.Linear(
    IMAGE_EMBEDDING_DIM,
    NUM_CLASSES
)

resnet.load_state_dict(
    torch.load(
        "../models/resnet50_best.pth",
        map_location=device
    )
)

resnet = resnet.to(device)

In [3]:
image_encoder = nn.Sequential(
    *list(resnet.children())[:-1]
)

image_encoder = image_encoder.to(device)
image_encoder.eval()

print("Image encoder loaded.")

Image encoder loaded.


2. Verify Image Embedding

In [5]:
%run ./02_Data_Preparation.ipynb

Device: mps
Train: (29901, 6)
Validation: (6435, 6)
Test: (6459, 6)
../data/raw/fashion-product-images-small/images/15970.jpg
Exists: True
Image size: (60, 80)
Image mode: RGB
Image mode: RGB
Tensor shape: torch.Size([3, 224, 224])
Tensor dtype: torch.float32
Tenosr min: -2.0836544036865234
Tensor max: 2.640000104904175
Dataset size: 29901
Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: tensor(17)
Label dtype: torch.int64
index 0: shape=torch.Size([3, 224, 224]), label=17
index 100: shape=torch.Size([3, 224, 224]), label=17
index 1000: shape=torch.Size([3, 224, 224]), label=13
index 5000: shape=torch.Size([3, 224, 224]), label=17
index 10000: shape=torch.Size([3, 224, 224]), label=0
Train datasets: 29901
Validation datasets: 6435
Test dataset: 6459
Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Images dtype: torch.float32
Label dtype: torch.int64
Labels: tensor([17, 17, 19, 17, 16, 17, 15, 15, 17, 15, 15,  0, 18, 17, 15, 17,  8, 11,


Text: Turtle Check Men Navy Blue Shirt
Input IDs shape: torch.Size([1, 32])
Attention mask shape: torch.Size([1, 32])
['[CLS]', 'turtle', 'check', 'men', 'navy', 'blue', 'shirt', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])
Orignal : Turtle Check Men Navy Blue Shirt
Decoded : turtle check men navy blue shirt
Input IDs shape: torch.Size([8, 32])
Attention mask shape: torch.Size([8, 32])
Maximum token length:  30
Texts with > 32 tokens: 0
Multimodal train dataset: 29901
dict_keys(['image', 'input_ids', 'attention_mask', 'label'])
Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Input IDs shape: torch.Size([32])
Input IDs dtype: torch.int64
Attention mask shape: torch.Size([32])
At

In [6]:
images, labels = next(
    iter(test_loader)
)

images = images.to(device)

with torch.no_grad():
    image_features = image_encoder(images)

image_features = torch.flatten(
    image_features,
    start_dim=1
)

print("Image features:", image_features.shape)

Image features: torch.Size([32, 2048])


3. Load the Trained Text Encoder

In [7]:
from transformers import AutoModel

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 32


class DistilBERTTextEncoder(nn.Module):

    def __init__(
        self,
        model_name="distilbert-base-uncased"
    ):
        super().__init__()

        self.transformer = AutoModel.from_pretrained(
            model_name
        )

        self.hidden_size = (
            self.transformer.config.hidden_size
        )

    def masked_mean_pooling(
        self,
        last_hidden_state,
        attention_mask
    ):
        mask = attention_mask.unsqueeze(-1).expand(
            last_hidden_state.size()
        ).float()

        summed_embeddings = torch.sum(
            last_hidden_state * mask,
            dim=1
        )

        token_count = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        return summed_embeddings / token_count

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        text_embedding = self.masked_mean_pooling(
            outputs.last_hidden_state,
            attention_mask
        )

        return text_embedding

In [8]:
class TextClassifier(nn.Module):

    def __init__(
        self,
        model_name="distilbert-base-uncased",
        num_classes=20
    ):
        super().__init__()

        self.text_encoder = DistilBERTTextEncoder(
            model_name
        )

        self.classifier = nn.Linear(
            self.text_encoder.hidden_size,
            num_classes
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        text_embedding = self.text_encoder(
            input_ids,
            attention_mask
        )

        logits = self.classifier(
            text_embedding
        )

        return logits

In [9]:
text_classifier = TextClassifier(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES
)

text_classifier.load_state_dict(
    torch.load(
        "../models/distilbert_text_classifier_best.pth",
        map_location=device
    )
)

text_classifier = text_classifier.to(device)
text_classifier.eval()

print("Text model loaded.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text model loaded.


4. Extract the Text Encoder

In [10]:
text_encoder = text_classifier.text_encoder

text_encoder = text_encoder.to(device)
text_encoder.eval()

print(
    "Text embedding dimension:",
    text_encoder.hidden_size
)

Text embedding dimension: 768


In [11]:
batch = next(
    iter(multimodal_test_loader)
)

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

with torch.no_grad():

    text_features = text_encoder(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("Text features:", text_features.shape)

Text features: torch.Size([32, 768])


5. Create the Multimodal Fusion Model

In [12]:
class MultimodalClassifier(nn.Module):

    def __init__(
        self,
        image_encoder,
        text_encoder,
        num_classes=20
    ):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Image projection
        self.image_projection = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU()
        )

        # Text projection
        self.text_projection = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU()
        )

        # Fusion classifier
        self.fusion = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        images,
        input_ids,
        attention_mask
    ):

        # Image features
        image_features = self.image_encoder(
            images
        )

        image_features = torch.flatten(
            image_features,
            start_dim=1
        )

        # Text features
        text_features = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Project both modalities
        image_features = self.image_projection(
            image_features
        )

        text_features = self.text_projection(
            text_features
        )

        # Concatenate
        fused_features = torch.cat(
            [image_features, text_features],
            dim=1
        )

        # Final classification
        logits = self.fusion(
            fused_features
        )

        return logits

6. Freeze Both Pretrained Encoder

In [13]:
for param in image_encoder.parameters():
    param.requires_grad = False

for param in text_encoder.parameters():
    param.requires_grad = False

7. Create the Multimodal Model

In [14]:
multimodal_model = MultimodalClassifier(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    num_classes=NUM_CLASSES
)

multimodal_model = multimodal_model.to(device)

print(multimodal_model)

MultimodalClassifier(
  (image_encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=Tru

In [15]:
total_params = sum(
    p.numel()
    for p in multimodal_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in multimodal_model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 91581268
Trainable parameters: 1710356


In [16]:
print("\nTrainable layers:")

for name, param in multimodal_model.named_parameters():

    if param.requires_grad:
        print(
            name,
            param.shape
        )


Trainable layers:
image_projection.0.weight torch.Size([512, 2048])
image_projection.0.bias torch.Size([512])
text_projection.0.weight torch.Size([512, 768])
text_projection.0.bias torch.Size([512])
fusion.0.weight torch.Size([256, 1024])
fusion.0.bias torch.Size([256])
fusion.3.weight torch.Size([20, 256])
fusion.3.bias torch.Size([20])


8. Test the Complete Multimodal Forward Pass

In [17]:
batch = next(
    iter(multimodal_test_loader)
)

images = batch["image"].to(device)
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["label"].to(device)

with torch.no_grad():

    logits = multimodal_model(
        images=images,
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("Image shape:", images.shape)
print("Input IDs shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)
print("Labels shape:", labels.shape)
print("Logits shape:", logits.shape)

Image shape: torch.Size([32, 3, 224, 224])
Input IDs shape: torch.Size([32, 32])
Attention mask shape: torch.Size([32, 32])
Labels shape: torch.Size([32])
Logits shape: torch.Size([32, 20])


### Multimodal Architecture Design & Encoder Integration

- Integrated trained ResNet-50 image encoder
- Integrated trained DistilBERT text encoder
- Image representation: 2048 dimensions
- Text representation: 768 dimensions
- Image projection: 2048 → 512
- Text projection: 768 → 512
- Fused representation: 1024 dimensions
- Fusion layer: 1024 → 256
- Dropout: 0.3
- Classification layer: 256 → 20
- ResNet-50 backbone frozen
- DistilBERT backbone frozen
- Projection and fusion layers trainable
- Total parameters: 91,581,268
- Trainable parameters: 1,710,356
- Verified complete multimodal forward pass
- Final logits shape: [32, 20]